In [2]:
import os
import librosa
import numpy as np

ravdess_path = r"C:\Users\Thimathi\source\Aurevia\data\raw\RAVDEES"

X = []
y = []

burnout_emotions = ["04", "05", "06", "07"]

for actor_folder in os.listdir(ravdess_path):
    actor_path = os.path.join(ravdess_path, actor_folder)

    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                file_path = os.path.join(actor_path, file)

                # Extract emotion ID from filename
                emotion = file.split("-")[2]

                # Convert to binary label
                label = 1 if emotion in burnout_emotions else 0

                # Load audio
                audio, sr = librosa.load(file_path, sr=None)

                # Extract MFCC
                mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
                mfcc_scaled = np.mean(mfcc.T, axis=0)

                X.append(mfcc_scaled)
                y.append(label)

print("Total samples:", len(X))
print("Labels:", len(y))

c:\Users\Thimathi\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total samples: 1440
Labels: 1440


In [3]:
from collections import Counter
print(Counter(y))

Counter({1: 768, 0: 672})


In [4]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.array(X)
y = np.array(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

Train: (1152, 40)
Validation: (288, 40)


In [5]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

In [6]:
import torch.nn as nn

class VoiceModel(nn.Module):
    def __init__(self):
        super(VoiceModel, self).__init__()
        self.fc1 = nn.Linear(40, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = VoiceModel()

In [7]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [8]:
num_epochs = 30
batch_size = 32

for epoch in range(num_epochs):
    model.train()
    
    for i in range(0, len(X_train), batch_size):
        xb = X_train[i:i+batch_size]
        yb = y_train[i:i+batch_size]
        
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    with torch.no_grad():
        outputs = model(X_val)
        _, preds = torch.max(outputs, 1)
        accuracy = (preds == y_val).float().mean()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Val Accuracy: {accuracy:.4f}")

Epoch 1, Loss: 1.8726, Val Accuracy: 0.4688
Epoch 2, Loss: 1.5602, Val Accuracy: 0.6007
Epoch 3, Loss: 0.9479, Val Accuracy: 0.5556
Epoch 4, Loss: 1.0839, Val Accuracy: 0.5799
Epoch 5, Loss: 1.2531, Val Accuracy: 0.4722
Epoch 6, Loss: 1.1424, Val Accuracy: 0.4757
Epoch 7, Loss: 0.8283, Val Accuracy: 0.5243
Epoch 8, Loss: 1.0076, Val Accuracy: 0.5694
Epoch 9, Loss: 0.8925, Val Accuracy: 0.5694
Epoch 10, Loss: 0.9260, Val Accuracy: 0.5729
Epoch 11, Loss: 0.8889, Val Accuracy: 0.5451
Epoch 12, Loss: 0.8739, Val Accuracy: 0.5590
Epoch 13, Loss: 0.9719, Val Accuracy: 0.5521
Epoch 14, Loss: 0.8469, Val Accuracy: 0.5729
Epoch 15, Loss: 0.8743, Val Accuracy: 0.5764
Epoch 16, Loss: 0.8318, Val Accuracy: 0.5590
Epoch 17, Loss: 0.8609, Val Accuracy: 0.5764
Epoch 18, Loss: 0.7349, Val Accuracy: 0.5938
Epoch 19, Loss: 0.8549, Val Accuracy: 0.5903
Epoch 20, Loss: 0.8088, Val Accuracy: 0.6181
Epoch 21, Loss: 0.7956, Val Accuracy: 0.5764
Epoch 22, Loss: 0.7996, Val Accuracy: 0.6076
Epoch 23, Loss: 0.7

In [10]:
import os
import librosa
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim

# =========================
# 1. LOAD DATA
# =========================

ravdess_path = r"C:\Users\Thimathi\source\Aurevia\data\raw\RAVDEES"

X = []
y = []

burnout_emotions = ["04", "05", "06", "07"]

for actor_folder in os.listdir(ravdess_path):
    actor_path = os.path.join(ravdess_path, actor_folder)

    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):

                file_path = os.path.join(actor_path, file)

                # Extract emotion ID
                emotion = file.split("-")[2]
                label = 1 if emotion in burnout_emotions else 0

                # Load audio
                audio, sr = librosa.load(file_path, sr=None)

                # Extract MFCC
                mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)

                # Pad / Truncate to fixed length (174)
                if mfcc.shape[1] < 174:
                    pad_width = 174 - mfcc.shape[1]
                    mfcc = np.pad(mfcc, ((0,0),(0,pad_width)), mode='constant')
                else:
                    mfcc = mfcc[:, :174]

                X.append(mfcc)
                y.append(label)

print("Total samples:", len(X))
print("Label distribution:", Counter(y))

# =========================
# 2. PREPARE DATA
# =========================

X = np.array(X)
y = np.array(y)

# Add channel dimension for CNN
X = X[:, np.newaxis, :, :]  # (samples, 1, 40, 174)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert to Torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

# =========================
# 3. DEFINE CNN MODEL
# =========================

class VoiceCNN(nn.Module):
    def __init__(self):
        super(VoiceCNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(0.3)

        # Dynamically calculate FC input size
        self._to_linear = None
        self.convs(torch.randn(1,1,40,174))

        self.fc1 = nn.Linear(self._to_linear, 128)
        self.fc2 = nn.Linear(128, 2)

    def convs(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))

        if self._to_linear is None:
            self._to_linear = x.view(1, -1).shape[1]

        return x

    def forward(self, x):
        x = self.convs(x)
        x = x.reshape(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = VoiceCNN()

# =========================
# 4. TRAINING SETUP
# =========================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 30
batch_size = 32

# =========================
# 5. TRAIN LOOP
# =========================

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for i in range(0, len(X_train), batch_size):
        xb = X_train[i:i+batch_size]
        yb = y_train[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    model.eval()
    with torch.no_grad():
        outputs = model(X_val)
        _, preds = torch.max(outputs, 1)
        accuracy = (preds == y_val).float().mean()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val Accuracy: {accuracy:.4f}")

# =========================
# 6. SAVE MODEL
# =========================

torch.save(model.state_dict(), "burnout_voice_model.pth")

print("Training Complete ✅")

Total samples: 1440
Label distribution: Counter({1: 768, 0: 672})
Epoch 1, Loss: 234.7388, Val Accuracy: 0.5451
Epoch 2, Loss: 24.5454, Val Accuracy: 0.6042
Epoch 3, Loss: 22.3174, Val Accuracy: 0.6215
Epoch 4, Loss: 19.0929, Val Accuracy: 0.5868
Epoch 5, Loss: 14.9722, Val Accuracy: 0.6215
Epoch 6, Loss: 11.0593, Val Accuracy: 0.6076
Epoch 7, Loss: 9.0551, Val Accuracy: 0.5938
Epoch 8, Loss: 7.7527, Val Accuracy: 0.6250
Epoch 9, Loss: 4.6261, Val Accuracy: 0.6215
Epoch 10, Loss: 2.6299, Val Accuracy: 0.6597
Epoch 11, Loss: 2.3628, Val Accuracy: 0.6389
Epoch 12, Loss: 1.9871, Val Accuracy: 0.6354
Epoch 13, Loss: 1.7135, Val Accuracy: 0.6667
Epoch 14, Loss: 1.3999, Val Accuracy: 0.6354
Epoch 15, Loss: 1.1302, Val Accuracy: 0.6354
Epoch 16, Loss: 0.9775, Val Accuracy: 0.6250
Epoch 17, Loss: 0.5711, Val Accuracy: 0.6319
Epoch 18, Loss: 0.4420, Val Accuracy: 0.6215
Epoch 19, Loss: 0.4332, Val Accuracy: 0.6181
Epoch 20, Loss: 0.2078, Val Accuracy: 0.6285
Epoch 21, Loss: 0.2418, Val Accuracy

In [15]:
import os
import librosa
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
import wandb

# =========================
# 1. CONFIG
# =========================

ravdess_path = r"C:\Users\Thimathi\source\Aurevia\data\raw\RAVDEES"

burnout_emotions = ["04", "05", "06", "07"]

NUM_EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.001
PATIENCE = 7

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 2. DATA AUGMENTATION
# =========================

def augment_audio(audio, sr):
    augmented = []

    # Add noise
    noise = np.random.randn(len(audio))
    audio_noise = audio + 0.005 * noise
    augmented.append(audio_noise)

    # Pitch shift
    audio_pitch = librosa.effects.pitch_shift(audio, sr=sr, n_steps=2)
    augmented.append(audio_pitch)

    return augmented

# =========================
# 3. LOAD DATA
# =========================

X = []
y = []

for actor_folder in os.listdir(ravdess_path):
    actor_path = os.path.join(ravdess_path, actor_folder)

    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):

                file_path = os.path.join(actor_path, file)
                emotion = file.split("-")[2]
                label = 1 if emotion in burnout_emotions else 0

                audio, sr = librosa.load(file_path, sr=None)

                samples = [audio] + augment_audio(audio, sr)

                for sample in samples:

                    mfcc = librosa.feature.mfcc(
                        y=sample,
                        sr=sr,
                        n_mfcc=40
                    )

                    if mfcc.shape[1] < 174:
                        pad = 174 - mfcc.shape[1]
                        mfcc = np.pad(mfcc, ((0,0),(0,pad)), mode='constant')
                    else:
                        mfcc = mfcc[:, :174]

                    X.append(mfcc)
                    y.append(label)

print("Total samples:", len(X))
print("Label distribution:", Counter(y))

X = np.array(X)
y = np.array(y)

X = X[:, np.newaxis, :, :]

# =========================
# 4. TRAIN/VAL SPLIT
# =========================

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================
# 5. SCALING (TRAIN ONLY FIT)
# =========================

scaler = StandardScaler()

X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)

scaler.fit(X_train_flat)

X_train = scaler.transform(X_train_flat).reshape(X_train.shape)
X_val = scaler.transform(X_val_flat).reshape(X_val.shape)

# =========================
# 6. TORCH TENSORS
# =========================

X_train = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
X_val = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
y_train = torch.tensor(y_train, dtype=torch.long).to(DEVICE)
y_val = torch.tensor(y_val, dtype=torch.long).to(DEVICE)

# =========================
# 7. MODEL
# =========================

class VoiceCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self._to_linear = None
        self._get_conv_output()

        self.classifier = nn.Sequential(
            nn.Linear(self._to_linear, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )

    def _get_conv_output(self):
        x = torch.randn(1,1,40,174)
        x = self.features(x)
        self._to_linear = x.view(1,-1).shape[1]

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

model = VoiceCNN().to(DEVICE)

# =========================
# 8. TRAINING SETUP
# =========================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

# =========================
# 9. WANDB INIT
# =========================

wandb.init(project="burnout-voice-research")

wandb.config.update({
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "architecture": "Deep CNN + BatchNorm",
    "dataset": "RAVDESS"
})

wandb.watch(model)

# =========================
# 10. TRAIN LOOP
# =========================

best_acc = 0
early_stop_counter = 0

for epoch in range(NUM_EPOCHS):

    model.train()
    total_loss = 0

    for i in range(0, len(X_train), BATCH_SIZE):

        xb = X_train[i:i+BATCH_SIZE]
        yb = y_train[i:i+BATCH_SIZE]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        outputs = model(X_val)
        _, preds = torch.max(outputs, 1)
        val_acc = (preds == y_val).float().mean().item()

    scheduler.step(val_acc)

    wandb.log({
        "epoch": epoch+1,
        "train_loss": total_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    if early_stop_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

# =========================
# 11. FINAL EVALUATION
# =========================

model.load_state_dict(torch.load("best_model.pth"))
model.eval()

with torch.no_grad():
    outputs = model(X_val)
    _, preds = torch.max(outputs, 1)

cm = confusion_matrix(y_val.cpu(), preds.cpu())
report = classification_report(y_val.cpu(), preds.cpu())

print("Confusion Matrix:\n", cm)
print("Classification Report:\n", report)

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_val.cpu().numpy(),
        preds=preds.cpu().numpy()
    )
})

wandb.save("best_model.pth")

print("Training Complete ✅")

Total samples: 4320
Label distribution: Counter({1: 2304, 0: 2016})


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Thimathi\_netrc.
wandb: Currently logged in as: chathuradissanayake274 (chathuradissanayake274-chathura) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1, Loss: 148.9811, Val Acc: 0.5764
Epoch 2, Loss: 73.5307, Val Acc: 0.5602
Epoch 3, Loss: 72.4066, Val Acc: 0.5972
Epoch 4, Loss: 70.9558, Val Acc: 0.6308
Epoch 5, Loss: 69.2266, Val Acc: 0.6609
Epoch 6, Loss: 68.4311, Val Acc: 0.6562
Epoch 7, Loss: 65.4639, Val Acc: 0.6644
Epoch 8, Loss: 64.1150, Val Acc: 0.6875
Epoch 9, Loss: 61.5260, Val Acc: 0.7014
Epoch 10, Loss: 59.3751, Val Acc: 0.7072
Epoch 11, Loss: 57.1963, Val Acc: 0.7373
Epoch 12, Loss: 55.6097, Val Acc: 0.7373
Epoch 13, Loss: 53.8310, Val Acc: 0.7558
Epoch 14, Loss: 51.5982, Val Acc: 0.7708
Epoch 15, Loss: 49.2984, Val Acc: 0.7396
Epoch 16, Loss: 47.7630, Val Acc: 0.7627
Epoch 17, Loss: 45.2152, Val Acc: 0.7720
Epoch 18, Loss: 41.3885, Val Acc: 0.7396
Epoch 19, Loss: 41.7480, Val Acc: 0.7778
Epoch 20, Loss: 38.2519, Val Acc: 0.7824
Epoch 21, Loss: 38.3722, Val Acc: 0.7558
Epoch 22, Loss: 34.8303, Val Acc: 0.7535
Epoch 23, Loss: 30.1584, Val Acc: 0.7766
Epoch 24, Loss: 31.8144, Val Acc: 0.7917
Epoch 25, Loss: 28.7405,

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Training Complete ✅
